## **Fixed-Effects & Neutral-Baseline Decomposition with Bootstrap**

**Purpose:**
This notebook implements two complementary decompositions of SES-related differences in essay scores:

1. **Fixed-Effects (FE) Decomposition** (Part A) — Estimates a two-way FE model to decompose SES gaps into *content*, *style*, and *other* components.
2. **Neutral-Baseline Decomposition** (Part B) — Uses a neutral GPT rewrite as the reference style to split the SES gap into *content*, *style premium*, and *scorer tilt*.

Both decompositions include clustered bootstrap confidence intervals (500 replications, essay-level clusters).

**How to Use:**
- **Part A cells only** → FE decomposition, bootstrap, tables, and figures.
- **Part B cells only** → Neutral-baseline decomposition, bootstrap, tables, and figures.
- **Run all** → Both decompositions end-to-end.

**Outputs:**
- `decomp_rows_with_fe.csv` → Row-level data with FE components for all essays and rewrites.
- `decomposition_bootstrap_summary_{full,demographics,prompts}.{csv,tex}` → FE decomposition tables.
- `decomp_rows_neutral.csv` → Row-level data with neutral-baseline components.
- `neutral_decomposition_bootstrap_summary_{full,demographics,prompts}.{csv,tex}` → Neutral decomposition tables.
- Distribution and scatter figures (PDF).

---
# Shared Setup

### Imports & Configuration

**Purpose:**
- Import all core Python libraries used throughout the notebook.
- Prepare tools for data handling (`pandas`, `numpy`), model estimation (`pyfixest`), progress visualization (`tqdm`),
  parallel computation (`joblib`), statistical modeling (`statsmodels`), and plotting (`matplotlib`, `seaborn`, `scipy`).
- Initialize a global random number generator (`rng`) with a fixed seed to ensure reproducibility of bootstrap samples.

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
from joblib import Parallel, delayed
import statsmodels.api as sm
from pyfixest.estimation import feols
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
from scipy.stats import pearsonr

# Reproducibility
rng = np.random.default_rng(42)

In [ ]:
out_path = "../../output/"
table_out_path = "../../output/tables/"

# Data paths
df_path_sat = "../../data/results/sat_full/data_sat_scored.csv"  # FE decomposition
df_path_neutral = "../../data/results/no_desc/no_desc_scored.csv"  # Neutral-baseline decomposition

# Figure settings
FIG_SIZE = (12, 6)
SAVE_DIR = "../../output/"

In [ ]:
def mean_mask(s: pd.Series, mask: pd.Series) -> float:
    """Compute conditional mean while safely handling missing values (NaN)."""
    s2 = s[mask & s.notna()]
    return float(s2.mean()) if len(s2) else np.nan

### Shared Label Maps & Formatting

In [ ]:
GROUP_LABELS = {
    "Overall": "All",
    "All": "All",
    "race_white=True": "White",
    "race_white=False": "Non-White",
    "gender_male=True": "Male",
    "gender_male=False": "Female",
    "Grade=6.0": "Grade 6",
    "Grade=8.0": "Grade 8",
    "Grade=10.0": "Grade 10",
    "Grade=11.0": "Grade 11",
    "grade_level=6.0": "Grade 6",
    "grade_level=8.0": "Grade 8",
    "grade_level=10.0": "Grade 10",
    "grade_level=11.0": "Grade 11",
}

K_LABELS = {
    0: "Original text",
    1: "SAT 1",
    2: "SAT 2",
    3: "SAT 3",
    4: "SAT 4",
    5: "SAT 5",
    6: "SAT 6",
    7: "Standard GPT",
}


def format_est_se(row):
    pe = row["point_estimate"]
    se = row.get("std_error", np.nan)
    if pd.isna(pe):
        return ""
    if pd.isna(se):
        return f"{pe:.3f}"
    return f"\\shortstack{{{pe:.3f}\\\\({se:.3f})}}"


def build_summary_visual_table(summary_df, stat_labels, stat_order, group_labels=None):
    """Pivot summary_df into a visual table with group as rows, stat as columns."""
    if group_labels is None:
        group_labels = GROUP_LABELS

    display_df = summary_df.copy()
    display_df["estimate_se"] = display_df.apply(format_est_se, axis=1)
    display_df["group"] = display_df["group"].map(lambda x: group_labels.get(x, x))
    display_df["stat"] = display_df["stat"].map(lambda x: stat_labels.get(x, x))

    vt = display_df.pivot(index="group", columns="stat", values="estimate_se")

    stat_col_order = [stat_labels[s] for s in stat_order if stat_labels[s] in vt.columns]
    vt = vt.reindex(columns=stat_col_order)

    ordered_groups = [
        group_labels.get("Overall", "All"),
        "Male", "Female", "White", "Non-White",
        "Grade 6", "Grade 8", "Grade 10", "Grade 11",
    ]
    prompt_rows = sorted([idx for idx in vt.index if str(idx).startswith("Prompt")])
    other_rows = [idx for idx in vt.index if idx not in ordered_groups and idx not in prompt_rows]
    final_order = [g for g in ordered_groups if g in vt.index] + prompt_rows + other_rows
    vt = vt.loc[final_order].rename_axis(index=None, columns=None)
    return vt


def save_tables(visual_table, prefix, table_out_path):
    """Save full, demographics-only, and prompts-only tables as CSV and LaTeX."""
    demo_rows_order = [
        "All", "Male", "Female", "White", "Non-White",
        "Grade 6", "Grade 8", "Grade 10", "Grade 11",
    ]
    demo_rows = [r for r in demo_rows_order if r in visual_table.index]
    prompt_rows = [c for c in visual_table.index if str(c).startswith("Prompt")]

    full_table = visual_table.copy()
    demo_table = visual_table.loc[demo_rows].copy()
    prompt_table = visual_table.loc[prompt_rows].copy()

    for label, tbl in [("full", full_table), ("demographics", demo_table), ("prompts", prompt_table)]:
        tbl.to_csv(f"{table_out_path}{prefix}_bootstrap_summary_{label}.csv")
        tbl.to_latex(f"{table_out_path}{prefix}_bootstrap_summary_{label}.tex", escape=False)

    print(f"Saved {prefix} tables (full / demographics / prompts) to {table_out_path}")
    return full_table, demo_table, prompt_table

---
# Part A — Fixed-Effects (FE) Decomposition

### A1 — Load & Prepare SAT Data

**Purpose:**
- Load the scored dataset containing essay-level and rewrite-level information.
- Sort rows by `essay_id` and `k` so each essay's rewrites appear sequentially, ensuring consistent ordering for later fixed-effect estimation and bootstrapping.
- Reset the index to maintain a clean 0–N row index after sorting.

In [ ]:
df = pd.read_csv(df_path_sat, encoding="ISO-8859-1", index_col=0)
df = df.sort_values(["essay_id", "k"]).reset_index(drop=True)

### A2 — Fit Fixed-Effects Model and Extract Baseline Estimates

**Purpose:**
- Estimate a two-way fixed-effects model using `pyfixest` to control for essay-specific and prompt-specific variation.
- Model specification: `score_high_full ~ 1 | essay_id + k` (fitted on rewrites only, `k != 0`).
- Retrieve the estimated fixed effects from the model output (`fes`):
  - `fe_essay` → captures essay-level (content) differences.
  - `fe_k` → captures prompt-level (context) effects.
- Map these fixed effects back to each observation in the DataFrame for later decomposition.
- Compute residuals `u = score_high_full – fe_essay`, representing the style component once essay-level effects are removed.

In [ ]:
# FE regression (fitted on rewrites only, k != 0)
res = feols("score_high_full ~ 1 | essay_id + k", data=df[df['k'] != 0])
print(res.summary)

fes = res.fixef()
fe_essay_map = fes["C(essay_id)"]
fe_k_map = fes["C(k)"]

# Map back to rows
df["fe_essay"] = df["essay_id"].map(fe_essay_map)
df["fe_k"] = df["k"].map(fe_k_map)

# Style residual (after removing essay FE only)
df["u"] = df["score_high_full"] - df["fe_essay"]

# Compute diff column
df["diff"] = df["score_high_full"] - df["score_low_full"]

# Save DataFrame with FEs and residuals
df.to_csv(out_path + "decomp_rows_with_fe.csv", index=False)
print(f"Saved augmented dataset - {out_path}decomp_rows_with_fe.csv")

### A3 — FE Decomposition Function

**Purpose:**
- Compute the SES gap and its decomposition into **content**, **style**, and **others** using the FE outputs.
- Restrict the decomposition to **k == 0** for comparability.
- Return both **absolute gaps** and **shares** (each component divided by the total gap).

**Steps:**
- **A) Total gap (k==0):** `total_gap = mean(score_high_full | high-SES) – mean(score_low_full | low-SES)`.
- **B) Content gap (k==0):** `content_gap = mean(fe_essay | high-SES) – mean(fe_essay | low-SES)`.
- **C) Style gap (k==0):** `style_gap = mean(u | high-SES) – mean(u | low-SES)`.
- **D) Others gap (k==0, ranker-tilting on low-SES originals):** `others_gap = mean(score_high_full – score_low_full | low-SES)`.

In [ ]:
def compute_fe_decomposition(df_in: pd.DataFrame):
    df_ = df_in.copy()

    # --- A) total gap (k==0)
    mA = (df_["low_SES"] == 0) & (df_["k"] == 0) & df_["score_high_full"].notna()
    mB = (df_["low_SES"] == 1) & (df_["k"] == 0) & df_["score_low_full"].notna()
    tmp_vals = pd.Series(np.nan, index=df_.index)
    tmp_vals[mA] = df_.loc[mA, "score_high_full"].astype(float)
    tmp_vals[mB] = df_.loc[mB, "score_low_full"].astype(float)

    group = pd.Series(np.nan, index=df_.index)
    group[mA] = 0
    group[mB] = 1

    mean0 = mean_mask(tmp_vals, group == 0)
    mean1 = mean_mask(tmp_vals, group == 1)
    total_gap = mean0 - mean1

    # --- B) content gap (k==0), use essay FE
    mK0 = (df_["k"] == 0)
    content_mean0 = mean_mask(df_["fe_essay"], mK0 & (df_["low_SES"] == 0))
    content_mean1 = mean_mask(df_["fe_essay"], mK0 & (df_["low_SES"] == 1))
    content_gap = content_mean0 - content_mean1

    # --- C) style gap via residual u = score_high_full - fe_essay
    u_mean0 = mean_mask(df_["u"], mK0 & (df_["low_SES"] == 0))
    u_mean1 = mean_mask(df_["u"], mK0 & (df_["low_SES"] == 1))
    style_gap = u_mean0 - u_mean1

    # --- D) ranker-tilting (on low-SES originals at k==0)
    m_tilt = (df_["low_SES"] == 1) & (df_["k"] == 0)
    others_gap = mean_mask(df_["score_high_full"] - df_["score_low_full"], m_tilt)

    return {
        "total_gap": total_gap,
        "content_gap": content_gap,
        "style_gap": style_gap,
        "others_gap": others_gap,
        "share_content": content_gap / total_gap if pd.notna(total_gap) and total_gap != 0 else np.nan,
        "share_style": style_gap / total_gap if pd.notna(total_gap) and total_gap != 0 else np.nan,
        "share_others": others_gap / total_gap if pd.notna(total_gap) and total_gap != 0 else np.nan,
    }


fe_point_est = compute_fe_decomposition(df)
fe_point_est

In [ ]:
df.to_csv(out_path + "decomp_rows_with_fe.csv", index=False)

### A4 — FE Bootstrap (Clustered at essay_id; Parallelized, 500 Reps, 2.5–97.5% CI)

**Purpose:**
- Estimate uncertainty for the FE decomposition metrics using **clustered bootstrapping** at the essay level.
- Each bootstrap sample resamples essays (clusters) **with replacement**, keeping all their rewrites together.
- The process re-fits the fixed-effects model on each resample, computes the decomposition again,
  and aggregates results across 500 iterations to form **95% confidence intervals** and standard errors.

In [ ]:
B = 500  # number of bootstrap iterations


def run_fe_bootstrap_for_group(df_group: pd.DataFrame, group_label: str, B: int = 500, seed: int = 42):
    """
    Cluster bootstrap for the FE decomposition, restricted to df_group.
    Re-fits the FE model on each resample.
    Returns a list of rows (dicts) for this group, ready to concat into summary_df.
    """
    df_g = df_group.reset_index(drop=True)
    essay_to_idx_g = df_g.groupby("essay_id").indices
    essay_ids_g = np.array(list(essay_to_idx_g.keys()))
    n_clusters_g = len(essay_ids_g)

    if n_clusters_g < 2:
        print(f"Skipping group {group_label}: only {n_clusters_g} distinct essays.")
        return []

    rng = np.random.default_rng(seed)
    seeds = rng.integers(0, 2**32 - 1, size=B, dtype=np.uint64)

    def one_rep_group(s: int) -> dict:
        rng_local = np.random.default_rng(s)
        sampled = rng_local.choice(essay_ids_g, size=n_clusters_g, replace=True)
        idx = np.concatenate([essay_to_idx_g[eid] for eid in sampled])
        df_b = df_g.iloc[idx].copy()

        res_b = feols("score_high_full ~ 1 | essay_id + k", data=df_b[df_b['k'] != 0])
        fes_b = res_b.fixef()
        fe_essay_b = fes_b["C(essay_id)"]
        fe_k_b = fes_b["C(k)"]

        df_b["fe_essay"] = df_b["essay_id"].map(fe_essay_b)
        df_b["fe_k"] = df_b["k"].map(fe_k_b)
        df_b["u"] = df_b["score_high_full"] - df_b["fe_essay"]

        return compute_fe_decomposition(df_b)

    print(f"Running {B} bootstrap iterations for group: {group_label} (N={len(df_g)}, essays={n_clusters_g})")

    results = Parallel(n_jobs=-1, backend="loky")(
        delayed(one_rep_group)(int(s)) for s in tqdm(seeds)
    )

    boot_df_stats = pd.DataFrame(results)

    ci_bounds = {}
    std_errors = {}
    for col in boot_df_stats.columns:
        low, high = np.nanpercentile(boot_df_stats[col], [2.5, 97.5])
        ci_bounds[col] = (low, high)
        std_errors[col] = boot_df_stats[col].std(ddof=1)

    point_est = compute_fe_decomposition(df_g)

    rows = []
    for stat_name, pe in point_est.items():
        lo, hi = ci_bounds[stat_name]
        se = std_errors.get(stat_name, np.nan)
        rows.append({
            "group": group_label,
            "stat": stat_name,
            "point_estimate": round(pe, 3),
            "std_error": round(se, 3) if pd.notna(se) else np.nan,
            "ci_2.5": round(lo, 3),
            "ci_97.5": round(hi, 3),
        })

    return rows

In [ ]:
fe_all_rows = []

# Overall
fe_all_rows += run_fe_bootstrap_for_group(df, "Overall", B=B, seed=42)

# White / Non-White
fe_all_rows += run_fe_bootstrap_for_group(df[df["race_white"] == True],  "White",     B=B, seed=101)
fe_all_rows += run_fe_bootstrap_for_group(df[df["race_white"] == False], "Non-White", B=B, seed=102)

# Male / Female
fe_all_rows += run_fe_bootstrap_for_group(df[df["gender_male"] == True],  "Male",   B=B, seed=201)
fe_all_rows += run_fe_bootstrap_for_group(df[df["gender_male"] == False], "Female", B=B, seed=202)

# All different prompts in prompt_name
for pname in sorted(df["prompt_name"].dropna().unique()):
    mask = df["prompt_name"] == pname
    label = f"Prompt={pname}"
    fe_all_rows += run_fe_bootstrap_for_group(df[mask], label, B=B, seed=300 + hash(pname) % 1000)

# All different grades in grade_level
for g in sorted(df["grade_level"].dropna().unique()):
    mask = df["grade_level"] == g
    label = f"Grade={g}"
    fe_all_rows += run_fe_bootstrap_for_group(df[mask], label, B=B, seed=400 + int(g))

# Final combined table
fe_summary_df = pd.DataFrame(fe_all_rows)
fe_summary_df

### A5 — FE Summary Table (Point Estimates + Bootstrap CIs)

In [ ]:
FE_STAT_LABELS = {
    "total_gap": "Total Gap",
    "content_gap": "Content",
    "style_gap": "Style",
    "others_gap": "Other",
    "share_content": "Share Content",
    "share_style": "Share Style",
    "share_others": "Share Other",
}
FE_STAT_ORDER = list(FE_STAT_LABELS.keys())

fe_visual_table = build_summary_visual_table(fe_summary_df, FE_STAT_LABELS, FE_STAT_ORDER)
fe_visual_table

In [ ]:
fe_full, fe_demo, fe_prompts = save_tables(fe_visual_table, "decomposition", table_out_path)

### A6 — FE Decomposition Figures

In [ ]:
df_decomp = pd.read_csv(out_path + "decomp_rows_with_fe.csv")

#### Distribution of Essay Fixed Effects (Content Component)

In [ ]:
x = df_decomp["fe_essay"].dropna()

mean = x.mean()
se = x.std(ddof=1) / np.sqrt(len(x))
q1, q3 = np.percentile(x, [25, 75])

plt.figure(figsize=FIG_SIZE)
sns.histplot(x, bins=30, color="mediumseagreen", stat="density", edgecolor="white", linewidth=0.7, alpha=0.8)
plt.axvline(q1, color="darkslategray", linestyle="--", linewidth=1.5, label="Q1 / Q3")
plt.axvline(q3, color="darkslategray", linestyle="--", linewidth=1.5)
plt.axvline(mean, color="teal", linewidth=2.2, label="Mean")
plt.text(mean, plt.ylim()[1]*0.95, f"Mean = {mean:.3f} (SE = {se:.5f})", ha="center", va="top", fontsize=10, color="teal")
plt.xlabel("Essay fixed effect (\u03b1)")
plt.ylabel("Density")
sns.despine()
plt.tight_layout()
plt.savefig(f"{out_path}fixed_effect_distribution.pdf")
plt.show()

#### Content Component Distribution by SES

In [ ]:
df_plot = df_decomp.dropna(subset=["fe_essay"])
low_df = df_plot[df_plot["low_SES"] == 1]["fe_essay"]
high_df = df_plot[df_plot["low_SES"] == 0]["fe_essay"]

def describe_fe(x):
    return {
        "mean": x.mean(),
        "se": x.std(ddof=1) / np.sqrt(len(x)),
        "q1": np.percentile(x, 25),
        "q3": np.percentile(x, 75),
    }

stats_low = describe_fe(low_df)
stats_high = describe_fe(high_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

sns.histplot(high_df, bins=30, stat="density", color="#5DADE2", edgecolor="white", linewidth=0.6, alpha=0.8, ax=axes[0])
axes[0].axvline(stats_high["mean"], color="#154360", linewidth=2)
axes[0].set_title(f"High SES  (Mean = {stats_high['mean']:.3f}, SE = {stats_high['se']:.4f})", fontsize=12, color="#154360", pad=12)
axes[0].set_xlabel("Content Component")
axes[0].set_ylabel("Density")

sns.histplot(low_df, bins=30, stat="density", color="#F5B041", edgecolor="white", linewidth=0.6, alpha=0.8, ax=axes[1])
axes[1].axvline(stats_low["mean"], color="#6E2C00", linewidth=2)
axes[1].set_title(f"Low SES  (Mean = {stats_low['mean']:.3f}, SE = {stats_low['se']:.4f})", fontsize=12, color="#6E2C00", pad=12)
axes[1].set_xlabel("Content Component")
axes[1].set_ylabel("")

for ax in axes:
    sns.despine(ax=ax)
    ax.axhline(0, color="gray", lw=0.5)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}fixed_effect_distribution_by_SES_dual.pdf", bbox_inches="tight")
plt.show()

#### Style Residual Distribution (Overall)

In [ ]:
mask = (df_decomp["k"] == 0) & df_decomp["u"].notna()
u = df_decomp.loc[mask, "u"].astype(float)

mean = u.mean()
q1, med, q3 = np.percentile(u, [25, 50, 75])
iqr = q3 - q1
se = u.std(ddof=1) / np.sqrt(len(u))

plt.figure(figsize=FIG_SIZE)
sns.set_style("whitegrid")
sns.histplot(u, bins=40, stat="density", color="#C7B6F7", edgecolor="white", linewidth=0.7, alpha=0.9)

yl = plt.ylim()
plt.axvline(med, color="#4E3AA3", linewidth=2.4, zorder=4, label=f"SE = {se:.4f}")

top = yl[1]
for xline in (q1, med, q3):
    plt.plot([xline, xline], [top*0.985, top*0.995], color="#4E3AA3", linewidth=2, zorder=5, clip_on=False)

plt.text(mean, top*0.965, f"Mean = {mean:.4f} | SE = {se:.4f}", ha="center", va="top", fontsize=10, color="#4E3AA3")
plt.xlabel("Style residual (u = score_high \u2212 FE)")
plt.ylabel("Density")
sns.despine()
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}style_distribution.pdf")
plt.show()

#### Style Residual Distribution by SES

In [ ]:
mask = (df_decomp["k"] == 0) & df_decomp["u"].notna()
df_u = df_decomp.loc[mask, ["low_SES", "u"]].astype(float)

u_high = df_u[df_u["low_SES"] == 0]["u"]
u_low = df_u[df_u["low_SES"] == 1]["u"]

def describe_u(x):
    return {
        "mean": x.mean(),
        "se": x.std(ddof=1) / np.sqrt(len(x)),
        "q1": np.percentile(x, 25),
        "med": np.percentile(x, 50),
        "q3": np.percentile(x, 75),
    }

stats_high = describe_u(u_high)
stats_low = describe_u(u_low)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
sns.set_style("whitegrid")

sns.histplot(u_high, bins=40, stat="density", color="#84C7F9", edgecolor="white", linewidth=0.6, alpha=0.9, ax=axes[0])
axes[0].axvline(stats_high['mean'], color="#1B4F72", linestyle="-", linewidth=2.4, zorder=3)
axes[0].set_title(f"High SES  (Mean = {stats_high['mean']:.4f}, SE = {stats_high['se']:.4f})", fontsize=12, color="#154360", pad=12)
axes[0].set_xlabel("Style Component")
axes[0].set_ylabel("Density")

sns.histplot(u_low, bins=40, stat="density", color="#F5B041", edgecolor="white", linewidth=0.6, alpha=0.9, ax=axes[1])
axes[1].axvline(stats_low['mean'], color="#4E3AA3", linestyle="-", linewidth=2.4, zorder=3)
axes[1].set_title(f"Low SES  (Mean = {stats_low['mean']:.4f}, SE = {stats_low['se']:.4f})", fontsize=12, color="#4E3AA3", pad=12)
axes[1].set_xlabel("Style Component")
axes[1].set_ylabel("")

for ax in axes:
    sns.despine(ax=ax)
    ax.axhline(0, color="gray", lw=0.5)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}style_distribution_by_SES_dual.pdf", bbox_inches="tight")
plt.show()

#### Content vs Style Scatter by SES

In [ ]:
df_scatter = df_decomp.loc[
    (df_decomp["k"] == 0) &
    df_decomp["u"].notna() &
    df_decomp["fe_essay"].notna(),
    ["low_SES", "u", "fe_essay"]
].astype(float).copy()

df_high = df_scatter[df_scatter["low_SES"] == 0]
df_low = df_scatter[df_scatter["low_SES"] == 1]

sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

def panel(ax, d, title, point_color, line_color):
    sns.scatterplot(data=d, x="fe_essay", y="u", s=22, alpha=0.55, color=point_color, edgecolor=None, ax=ax)
    sns.regplot(data=d, x="fe_essay", y="u", scatter=False, ci=95, color=line_color, line_kws={"lw": 2}, ax=ax)
    r, p = pearsonr(d["fe_essay"].to_numpy(), d["u"].to_numpy())
    ax.set_title(f"{title}\nPearson r = {r:.3f}", fontsize=12)
    ax.set_xlabel("Content Component (fe_essay)")
    ax.set_ylabel("Style Component (u)")
    sns.despine(ax=ax)

panel(axes[0], df_high, "High SES", point_color="#5DADE2", line_color="#154360")
panel(axes[1], df_low, "Low SES", point_color="#F5B041", line_color="#6E2C00")

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}style_vs_content_scatter_by_SES_dual.pdf", bbox_inches="tight")
plt.show()

#### FE Stacked-Bar Decomposition Shares

In [ ]:
FE_COMP_ORDER = ["content", "style", "others"]
FE_COMP_COLORS = {"content": "#84C7F9", "style": "#C7B6F7", "others": "#B6E3C7"}


def _fe_compute_decomp(df_sub):
    m_orig = (df_sub["k"] == 0)
    m_hi = m_orig & (df_sub["low_SES"] == 0) & df_sub["score_high_full"].notna()
    m_lo = m_orig & (df_sub["low_SES"] == 1) & df_sub["score_low_full"].notna()
    if not (m_hi.any() and m_lo.any()):
        return pd.Series({c: np.nan for c in FE_COMP_ORDER})

    mean_hi = mean_mask(df_sub["score_high_full"], m_hi)
    mean_lo = mean_mask(df_sub["score_low_full"], m_lo)
    total_gap = mean_hi - mean_lo

    content_hi = mean_mask(df_sub["fe_essay"], m_orig & (df_sub["low_SES"] == 0))
    content_lo = mean_mask(df_sub["fe_essay"], m_orig & (df_sub["low_SES"] == 1))
    style_hi = mean_mask(df_sub["u"], m_orig & (df_sub["low_SES"] == 0))
    style_lo = mean_mask(df_sub["u"], m_orig & (df_sub["low_SES"] == 1))

    content_gap = content_hi - content_lo
    style_gap = style_hi - style_lo
    others_gap = total_gap - (content_gap + style_gap)

    if not total_gap:
        return pd.Series({c: np.nan for c in FE_COMP_ORDER})

    return pd.Series({
        "content": content_gap / total_gap,
        "style": style_gap / total_gap,
        "others": others_gap / total_gap,
    })


def fe_compute_decomp_shares(df_decomp):
    df_ = df_decomp.copy()
    groups = {"All": df_}
    for g in ["race_white", "gender_male"]:
        if g in df_.columns:
            groups[f"{g}=True"] = df_[df_[g] == True]
            groups[f"{g}=False"] = df_[df_[g] == False]
    if "grade_level" in df_.columns:
        for gl in sorted(df_["grade_level"].dropna().unique()):
            groups[f"grade_level={gl}"] = df_[df_["grade_level"] == gl]
    if "prompt_name" in df_.columns:
        for p in sorted(df_["prompt_name"].dropna().unique()):
            groups[f"Prompt: {p}"] = df_[df_["prompt_name"] == p]
    out = pd.DataFrame({k: _fe_compute_decomp(v) for k, v in groups.items()}).T
    return out.reindex(columns=FE_COMP_ORDER)


def plot_stacked(df_shares, comp_order, comp_colors, title=None, figsize=(10, 6),
                 xtick_rotation=0, save_path=None, label_map=None):
    if df_shares.empty:
        print("No data to plot.")
        return
    if label_map is None:
        label_map = GROUP_LABELS
    df_plot = df_shares.copy()
    df_plot.index = df_plot.index.map(lambda x: label_map.get(x, x))

    ax = df_plot[comp_order].plot(
        kind="bar", stacked=True,
        color=[comp_colors[c] for c in comp_order],
        edgecolor="white", linewidth=0.6, figsize=figsize,
    )
    ax.set_ylabel("Share of Total SES Gap")
    ax.set_xlabel("")
    ax.set_ylim(0, 1)
    ax.axhline(1, color="gray", lw=0.8, linestyle="--")
    if title:
        ax.set_title(title, pad=10)
    plt.xticks(rotation=xtick_rotation, ha="right" if xtick_rotation else "center")
    ax.legend(title="Component", loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()


def plot_overview(decomp_shares, comp_order, comp_colors, save_path=None, label_map=None):
    base = ["All", "race_white=True", "race_white=False", "gender_male=True", "gender_male=False"]
    grade_keys = sorted([i for i in decomp_shares.index if i.startswith("grade_level=")], key=lambda x: float(x.split("=")[1]))
    want = base + grade_keys
    overview = decomp_shares.reindex(want).dropna(how="all")
    plot_stacked(overview, comp_order, comp_colors, figsize=FIG_SIZE, xtick_rotation=45, save_path=save_path, label_map=label_map)


def plot_prompts(decomp_shares, comp_order, comp_colors, save_path=None, label_map=None, figsize=(10, 6)):
    prompts = decomp_shares.loc[decomp_shares.index.str.startswith("Prompt: ")].sort_index()
    plot_stacked(prompts, comp_order, comp_colors, figsize=figsize, xtick_rotation=45, save_path=save_path, label_map=label_map)

In [ ]:
fe_decomp_shares = fe_compute_decomp_shares(df_decomp)

plot_stacked(
    fe_decomp_shares, FE_COMP_ORDER, FE_COMP_COLORS,
    figsize=(20, 10), xtick_rotation=45,
    save_path=f"{SAVE_DIR}decomp_shares_all.pdf", label_map=GROUP_LABELS,
)

plot_overview(
    fe_decomp_shares, FE_COMP_ORDER, FE_COMP_COLORS,
    save_path=f"{SAVE_DIR}decomp_shares_overview.pdf", label_map=GROUP_LABELS,
)

plot_prompts(
    fe_decomp_shares, FE_COMP_ORDER, FE_COMP_COLORS,
    figsize=(20, 10),
    save_path=f"{SAVE_DIR}decomp_shares_prompts.pdf", label_map=GROUP_LABELS,
)

---
# Part B — Neutral-Baseline Decomposition

### B1 — Load Neutral-Rewrite Data

**Purpose:**
- Load the scored dataset that includes neutral GPT rewrites.
- Sort rows by `essay_id` and `k`.

In [ ]:
df_neu_raw = pd.read_csv(df_path_neutral, encoding="ISO-8859-1", index_col=0)
df_neu_raw = df_neu_raw.sort_values(["essay_id", "k"]).reset_index(drop=True)

### B2 — Build Baseline Dataset (Original + Neutral Rewrite)

**Purpose:**
- Use the neutral GPT rewrite (`k == 1`) as the reference style.
- Construct a per-essay dataset that keeps original scores from both scorers, the neutral high-scorer score, and derived components (neutral content, style premium, scorer tilt).
- Drop essays without a matching neutral rewrite (grades 9 and 12 are removed for consistency).

In [ ]:
def build_baseline_decomp_df(df_in: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare one row per essay with original scores (both scorers) and the neutral GPT rewrite
    scored by the high-SES scorer. Adds derived columns used in the baseline decomposition.
    """
    df_ = df_in.copy()

    df_orig = df_[df_["k"] == 0].copy()
    df_neu = df_[df_["k"] != 0].copy()

    df_orig = df_orig[~df_orig["grade_level"].isin([9.0, 12.0])]
    df_neu = df_neu[df_neu["essay_id"].isin(df_orig["essay_id"])]

    keep_cols = [
        "essay_id", "low_SES", "race_white", "gender_male",
        "prompt_name", "grade_level", "cv_fold",
        "score_high_full", "score_low_full",
    ]
    df_orig = df_orig[keep_cols].rename(columns={
        "score_high_full": "score_high_orig",
        "score_low_full": "score_low_orig",
    })

    df_neu = df_neu[["essay_id", "score_high_full"]].rename(columns={
        "score_high_full": "score_high_neu",
    })

    out = df_orig.merge(df_neu, on="essay_id", how="inner")

    out["content_neu"] = out["score_high_neu"]
    out["style_premium"] = out["score_high_orig"] - out["score_high_neu"]
    out["diff_high_low_orig"] = out["score_high_orig"] - out["score_low_orig"]
    out["k"] = 0

    return out


df_baseline = build_baseline_decomp_df(df_neu_raw)
print(f"Baseline dataset size: {len(df_baseline):,} essays")

df_baseline.to_csv(out_path + "decomp_rows_neutral.csv", index=False)
print(f"Saved baseline dataset -> {out_path}decomp_rows_neutral.csv")

### B3 — Neutral Decomposition Function

**Purpose:**
- Compute the SES gap and its baseline decomposition into **content**, **style premium**, and **scorer tilt**.
- Content: gap in neutral scores under the high-SES scorer.
- Style: difference in style premia relative to neutral between SES groups.
- Tilt: scorer-function tilt on low-SES essays.
- Total gap uses the observed scoring pair so that **total = content + style + tilt**.

In [ ]:
def compute_neutral_decomposition(df_in: pd.DataFrame):
    df0 = df_in.copy()

    mH = df0["low_SES"] == 0
    mL = df0["low_SES"] == 1

    mu_H_orig = mean_mask(df0["score_high_orig"], mH)
    mu_L_orig_high = mean_mask(df0["score_high_orig"], mL)
    mu_H_neu = mean_mask(df0["score_high_neu"], mH)
    mu_L_neu = mean_mask(df0["score_high_neu"], mL)
    mu_L_orig_low = mean_mask(df0["score_low_orig"], mL)

    total_gap = mu_H_orig - mu_L_orig_low
    content_gap = mu_H_neu - mu_L_neu
    style_gap = (mu_H_orig - mu_H_neu) - (mu_L_orig_high - mu_L_neu)
    tilt_gap = mu_L_orig_high - mu_L_orig_low

    denom = total_gap if pd.notna(total_gap) and total_gap != 0 else np.nan

    return {
        "total_gap": total_gap,
        "content_gap": content_gap,
        "style_gap": style_gap,
        "tilt_gap": tilt_gap,
        "share_content": content_gap / denom if pd.notna(denom) else np.nan,
        "share_style": style_gap / denom if pd.notna(denom) else np.nan,
        "share_tilt": tilt_gap / denom if pd.notna(denom) else np.nan,
    }


neu_point_est = compute_neutral_decomposition(df_baseline)
neu_point_est

### B4 — Neutral Bootstrap (Clustered at essay_id; Parallelized, 500 Reps, 2.5–97.5% CI)

**Purpose:**
- Cluster-bootstraps the baseline decomposition at the essay level.
- Each replication resamples essays (with all attached baseline columns), recomputes the decomposition, and stores percentile-based 95% CIs.
- No FE re-estimation needed for the neutral decomposition — only resampling and means.

In [ ]:
def run_neutral_bootstrap_for_group(df_group: pd.DataFrame, group_label: str, B: int = 500, seed: int = 42):
    """
    Cluster bootstrap for the neutral-baseline decomposition (one row per essay).
    """
    df_g = df_group.reset_index(drop=True)
    essay_to_idx_g = df_g.groupby("essay_id").indices
    essay_ids_g = np.array(list(essay_to_idx_g.keys()))
    n_clusters_g = len(essay_ids_g)

    if n_clusters_g < 2:
        print(f"Skipping group {group_label}: only {n_clusters_g} distinct essays.")
        return []

    rng = np.random.default_rng(seed)
    seeds = rng.integers(0, 2**32 - 1, size=B, dtype=np.uint64)

    def one_rep_group(s: int) -> dict:
        rng_local = np.random.default_rng(s)
        sampled = rng_local.choice(essay_ids_g, size=n_clusters_g, replace=True)
        idx = np.concatenate([essay_to_idx_g[eid] for eid in sampled])
        df_b = df_g.iloc[idx].copy()
        return compute_neutral_decomposition(df_b)

    print(f"Running {B} bootstrap iterations for group: {group_label} (N={len(df_g)}, essays={n_clusters_g})")

    results = Parallel(n_jobs=-1, backend="loky")(
        delayed(one_rep_group)(int(s)) for s in tqdm(seeds)
    )

    boot_df_stats = pd.DataFrame(results)

    ci_bounds = {}
    std_errors = {}
    for col in boot_df_stats.columns:
        low, high = np.nanpercentile(boot_df_stats[col], [2.5, 97.5])
        ci_bounds[col] = (low, high)
        std_errors[col] = boot_df_stats[col].std(ddof=1)

    point_est = compute_neutral_decomposition(df_g)

    rows = []
    for stat_name, pe in point_est.items():
        lo, hi = ci_bounds.get(stat_name, (np.nan, np.nan))
        se = std_errors.get(stat_name, np.nan)
        rows.append({
            "group": group_label,
            "stat": stat_name,
            "point_estimate": round(pe, 3),
            "std_error": round(se, 3) if pd.notna(se) else np.nan,
            "ci_2.5": round(lo, 3),
            "ci_97.5": round(hi, 3),
        })

    return rows

In [ ]:
neu_all_rows = []

# Overall
neu_all_rows += run_neutral_bootstrap_for_group(df_baseline, "Overall", B=B, seed=42)

# White / Non-White
neu_all_rows += run_neutral_bootstrap_for_group(df_baseline[df_baseline["race_white"] == True],  "White",     B=B, seed=101)
neu_all_rows += run_neutral_bootstrap_for_group(df_baseline[df_baseline["race_white"] == False], "Non-White", B=B, seed=102)

# Male / Female
neu_all_rows += run_neutral_bootstrap_for_group(df_baseline[df_baseline["gender_male"] == True],  "Male",   B=B, seed=201)
neu_all_rows += run_neutral_bootstrap_for_group(df_baseline[df_baseline["gender_male"] == False], "Female", B=B, seed=202)

# All different prompts
for pname in sorted(df_baseline["prompt_name"].dropna().unique()):
    mask = df_baseline["prompt_name"] == pname
    label = f"Prompt={pname}"
    neu_all_rows += run_neutral_bootstrap_for_group(df_baseline[mask], label, B=B, seed=300 + hash(pname) % 1000)

# All different grades
for g in sorted(df_baseline["grade_level"].dropna().unique()):
    mask = df_baseline["grade_level"] == g
    label = f"Grade={g}"
    neu_all_rows += run_neutral_bootstrap_for_group(df_baseline[mask], label, B=B, seed=400 + int(g))

neu_summary_df = pd.DataFrame(neu_all_rows)
neu_summary_df

### B5 — Neutral Summary Table (Point Estimates + Bootstrap CIs)

In [ ]:
NEU_STAT_LABELS = {
    "total_gap": "Total Gap",
    "content_gap": "Content",
    "style_gap": "Style",
    "tilt_gap": "Other",
    "share_content": "Share Content",
    "share_style": "Share Style",
    "share_tilt": "Share Other",
}
NEU_STAT_ORDER = list(NEU_STAT_LABELS.keys())

neu_visual_table = build_summary_visual_table(neu_summary_df, NEU_STAT_LABELS, NEU_STAT_ORDER)
neu_visual_table

In [ ]:
neu_full, neu_demo, neu_prompts = save_tables(neu_visual_table, "neutral_decomposition", table_out_path)

### B6 — Neutral Decomposition Figures

In [ ]:
df_neu_decomp = pd.read_csv(out_path + "decomp_rows_neutral.csv")

#### Neutral Stacked-Bar Decomposition Shares

In [ ]:
NEU_COMP_ORDER = ["content", "style", "tilt"]
NEU_COMP_COLORS = {"content": "#84C7F9", "style": "#C7B6F7", "tilt": "#B6E3C7"}


def _neu_compute_decomp(df_sub):
    mH = df_sub["low_SES"] == 0
    mL = df_sub["low_SES"] == 1
    if not (mH.any() and mL.any()):
        return pd.Series({c: np.nan for c in NEU_COMP_ORDER})

    mu_H_orig = mean_mask(df_sub["score_high_orig"], mH)
    mu_L_orig_high = mean_mask(df_sub["score_high_orig"], mL)
    mu_L_orig_low = mean_mask(df_sub["score_low_orig"], mL)
    mu_H_neu = mean_mask(df_sub["score_high_neu"], mH)
    mu_L_neu = mean_mask(df_sub["score_high_neu"], mL)

    total_gap = mu_H_orig - mu_L_orig_low
    if not total_gap:
        return pd.Series({c: np.nan for c in NEU_COMP_ORDER})

    content_gap = mu_H_neu - mu_L_neu
    style_gap = mean_mask(df_sub["style_premium"], mH) - mean_mask(df_sub["style_premium"], mL)
    tilt_gap = mean_mask(df_sub["diff_high_low_orig"], mL)

    return pd.Series({
        "content": content_gap / total_gap,
        "style": style_gap / total_gap,
        "tilt": tilt_gap / total_gap,
    })


def neu_compute_decomp_shares(df_decomp):
    df_ = df_decomp.copy()
    groups = {"All": df_}
    for g in ["race_white", "gender_male"]:
        if g in df_.columns:
            groups[f"{g}=True"] = df_[df_[g] == True]
            groups[f"{g}=False"] = df_[df_[g] == False]
    if "grade_level" in df_.columns:
        for gl in sorted(df_["grade_level"].dropna().unique()):
            groups[f"grade_level={gl}"] = df_[df_["grade_level"] == gl]
    if "prompt_name" in df_.columns:
        for p in sorted(df_["prompt_name"].dropna().unique()):
            groups[f"Prompt: {p}"] = df_[df_["prompt_name"] == p]
    out = pd.DataFrame({k: _neu_compute_decomp(v) for k, v in groups.items()}).T
    return out.reindex(columns=NEU_COMP_ORDER)

In [ ]:
neu_decomp_shares = neu_compute_decomp_shares(df_neu_decomp)

plot_stacked(
    neu_decomp_shares, NEU_COMP_ORDER, NEU_COMP_COLORS,
    figsize=(20, 10), xtick_rotation=45,
    save_path=f"{SAVE_DIR}neutral_decomp_shares_all.pdf", label_map=GROUP_LABELS,
)

plot_overview(
    neu_decomp_shares, NEU_COMP_ORDER, NEU_COMP_COLORS,
    save_path=f"{SAVE_DIR}neutral_decomp_shares_overview.pdf", label_map=GROUP_LABELS,
)

plot_prompts(
    neu_decomp_shares, NEU_COMP_ORDER, NEU_COMP_COLORS,
    figsize=(20, 10),
    save_path=f"{SAVE_DIR}neutral_decomp_shares_prompts.pdf", label_map=GROUP_LABELS,
)

#### Style Premium by Group and SES

Shows the style premium (orig - neutral under the high-SES scorer) separately for high-SES and low-SES writers, across the full sample, demographics, and grades. Bars are side-by-side with 95% CI error bars (mean +/- 1.96*SE).

In [ ]:
def compute_style_premium_levels(df_in: pd.DataFrame):
    groups = {"All": df_in}
    for g in ["race_white", "gender_male"]:
        if g in df_in.columns:
            groups[f"{g}=True"] = df_in[df_in[g] == True]
            groups[f"{g}=False"] = df_in[df_in[g] == False]
    if "grade_level" in df_in.columns:
        for gl in sorted(df_in["grade_level"].dropna().unique()):
            groups[f"grade_level={gl}"] = df_in[df_in["grade_level"] == gl]

    rows = []
    for name, sub in groups.items():
        sub = sub[sub["style_premium"].notna()]
        high = sub[sub["low_SES"] == 0]["style_premium"].astype(float)
        low = sub[sub["low_SES"] == 1]["style_premium"].astype(float)
        if len(high) == 0 or len(low) == 0:
            continue
        rows.append({
            "group": name,
            "mean_high": high.mean(),
            "se_high": high.std(ddof=1) / np.sqrt(len(high)),
            "mean_low": low.mean(),
            "se_low": low.std(ddof=1) / np.sqrt(len(low)),
            "n_high": len(high),
            "n_low": len(low),
        })
    return pd.DataFrame(rows)


def plot_style_premium_levels(df_stats, save_path=None, label_map=None):
    if df_stats.empty:
        print("No style premium data to plot.")
        return
    if label_map is None:
        label_map = GROUP_LABELS

    base_order = [
        "All", "race_white=True", "race_white=False",
        "gender_male=True", "gender_male=False",
    ]
    grade_keys = sorted(
        [g for g in df_stats["group"] if g.startswith("grade_level=")],
        key=lambda x: float(x.split("=")[1]),
    )
    order_map = {g: i for i, g in enumerate(base_order + grade_keys)}
    default_start = len(order_map)

    df_plot = df_stats.copy()
    df_plot["order"] = [order_map.get(g, default_start + i) for i, g in enumerate(df_plot["group"])]
    df_plot = df_plot.sort_values("order")
    df_plot["label"] = df_plot["group"].map(lambda x: label_map.get(x, x))

    x = np.arange(len(df_plot))
    width = 0.38

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(
        x - width / 2, df_plot["mean_low"],
        yerr=1.96 * df_plot["se_low"], width=width,
        color=NEU_COMP_COLORS.get("style", "#C7B6F7"),
        edgecolor="white", linewidth=0.8, capsize=6, alpha=0.9, label="Low SES",
    )
    ax.bar(
        x + width / 2, df_plot["mean_high"],
        yerr=1.96 * df_plot["se_high"], width=width,
        color=NEU_COMP_COLORS.get("content", "#84C7F9"),
        edgecolor="white", linewidth=0.8, capsize=6, alpha=0.9, label="High SES",
    )

    ax.axhline(0, color="gray", lw=0.8, linestyle="--")
    ax.set_ylabel("Style premium (orig \u2212 neutral) under S(H)")
    ax.set_xlabel("")
    ax.set_title("Style premium by SES and group", pad=10)
    ax.set_xticks(x)
    ax.set_xticklabels(df_plot["label"], rotation=45, ha="right")
    ax.legend(frameon=False)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()


style_premium_levels = compute_style_premium_levels(df_neu_decomp)
plot_style_premium_levels(
    style_premium_levels,
    save_path=f"{SAVE_DIR}style_premium_levels_by_group.pdf",
    label_map=GROUP_LABELS,
)